# Phase 4: Classical CV Baseline

Traditional image processing pipeline for defect detection.
Used as a comparison baseline against the GAN + EfficientNet approach.

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.cv_baseline import (
    preprocess, threshold_detection, edge_detection,
    morphological_detection, texture_analysis,
    analyze_contours, classify, detect, evaluate_folder
)
from src.utils.config import load_config

config = load_config('../config.yaml')
print('CV baseline loaded OK')

## Step 1: Visualize Each CV Stage on a Sample Image

In [ ]:
# Change this to any image in your dataset
IMAGE_PATH = f"{config.data.raw_dir}/bottle/test/broken_large/000.png"

img_bgr = cv2.imread(IMAGE_PATH)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

img_bgr_r, gray = preprocess(img_bgr, size=256)
binary   = threshold_detection(gray)
edges    = edge_detection(gray)
morph    = morphological_detection(binary)
texture  = texture_analysis(gray)
combined = cv2.bitwise_or(morph, texture)

titles = ['Original', 'Grayscale + CLAHE', 'Otsu Threshold',
          'Canny Edges', 'Morphological', 'Combined Mask']
images = [
    cv2.cvtColor(img_bgr_r, cv2.COLOR_BGR2RGB),
    gray, binary, edges, morph, combined
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for ax, title, img in zip(axes.flat, titles, images):
    cmap = 'gray' if len(img.shape) == 2 else None
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=12)
    ax.axis('off')

plt.suptitle('CV Pipeline Stages', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/cv_stages.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/cv_stages.png')

## Step 2: Contour Detection & Bounding Boxes

In [ ]:
defects = analyze_contours(combined, min_area=100)
result  = classify(defects, image_area=256*256)

annotated = cv2.cvtColor(img_bgr_r, cv2.COLOR_BGR2RGB).copy()
for d in defects:
    x, y, w, h = d['bbox']
    cv2.rectangle(annotated, (x, y), (x+w, y+h), (255, 0, 0), 2)
    cv2.circle(annotated, d['centroid'], 5, (0, 255, 0), -1)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(combined, cmap='gray')
axes[0].set_title('Combined Defect Mask')
axes[0].axis('off')

axes[1].imshow(annotated)
axes[1].set_title(f"Detected: {result['label'].upper()}  ({result['num_defects']} regions)")
axes[1].axis('off')

plt.tight_layout()
plt.savefig('../outputs/cv_contours.png', dpi=150, bbox_inches='tight')
plt.show()

print('Result:', result)

## Step 3: Compare Normal vs Defective

In [ ]:
normal_path  = f"{config.data.raw_dir}/bottle/test/good/000.png"
defect_path  = f"{config.data.raw_dir}/bottle/test/broken_large/000.png"

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for row, (path, title) in enumerate([(normal_path, 'NORMAL'), (defect_path, 'DEFECTIVE')]):
    img = cv2.imread(path)
    img_r, gray_r = preprocess(img, 256)
    binary_r = threshold_detection(gray_r)
    morph_r  = morphological_detection(binary_r)
    texture_r = texture_analysis(gray_r)
    combined_r = cv2.bitwise_or(morph_r, texture_r)
    defects_r = analyze_contours(combined_r)
    result_r  = classify(defects_r, 256*256)

    axes[row, 0].imshow(cv2.cvtColor(img_r, cv2.COLOR_BGR2RGB))
    axes[row, 0].set_title(f'{title} — Original')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(combined_r, cmap='gray')
    axes[row, 1].set_title('Combined Mask')
    axes[row, 1].axis('off')

    ann = cv2.cvtColor(img_r, cv2.COLOR_BGR2RGB).copy()
    for d in defects_r:
        x, y, w, h = d['bbox']
        cv2.rectangle(ann, (x,y), (x+w,y+h), (255,0,0), 2)
    axes[row, 2].imshow(ann)
    axes[row, 2].set_title(f"Pred: {result_r['label'].upper()}  conf={result_r['confidence']}")
    axes[row, 2].axis('off')

plt.suptitle('Normal vs Defective — CV Pipeline', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4: Evaluate CV Baseline on Test Folder

In [ ]:
import pandas as pd

category = 'bottle'
base = Path(config.data.raw_dir) / category / 'test'

# Evaluate normal images (label=0)
normal_metrics = evaluate_folder(str(base / 'good'), label=0)

# Evaluate defective images (label=1) — pick first defect subfolder
defect_dirs = [d for d in base.iterdir() if d.is_dir() and d.name != 'good']
defect_metrics = evaluate_folder(str(defect_dirs[0]), label=1) if defect_dirs else {}

print('=== NORMAL IMAGES ===')
for k, v in normal_metrics.items(): print(f'  {k}: {v}')

print('\n=== DEFECTIVE IMAGES ===')
for k, v in defect_metrics.items(): print(f'  {k}: {v}')

## Step 5: CV vs Deep Learning — Comparison Table

In [ ]:
import pandas as pd

# Fill in your actual DL results after training
comparison = pd.DataFrame([
    {'Method': 'Classical CV (Otsu+Morph)',  'Accuracy': normal_metrics.get('accuracy', '-'), 'F1': normal_metrics.get('f1', '-'), 'Notes': 'No training needed'},
    {'Method': 'EfficientNet-B2 (no GAN)',   'Accuracy': '-', 'F1': '-', 'Notes': 'Fill after training'},
    {'Method': 'EfficientNet-B2 + GAN Aug',  'Accuracy': '-', 'F1': '-', 'Notes': 'Fill after training'},
])

print(comparison.to_string(index=False))